# Sentiment analysis using python library

In [1]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [2]:
from textblob import TextBlob

In [3]:
def sentiment_analysis(review):
    analysis = TextBlob(review)
    if analysis.sentiment.polarity > 0:
        result = 'Positive'
    elif analysis.sentiment.polarity == 0:
        result = 'Neutral'
    else:
        result = 'Negative'
    return result

In [4]:
reviews = ['I love this product. i will buy again','I hate this thing', 'product is okay. it could be better', 
           'the product is not good, but I can use']

for review in reviews:
    print(f'{review} : {sentiment_analysis(review)}')

I love this product. i will buy again : Positive
I hate this thing : Negative
product is okay. it could be better : Positive
the product is not good, but I can use : Negative


# Load necessary python libraries

In [5]:
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime
import os
import dotenv

# Load environment variables

In [6]:
dotenv.load_dotenv()
mssql_db = os.getenv('mssql')

# Create database connection

In [7]:
# create database connection
engine = create_engine(mssql_db)
conn = engine.connect()

# Process yelp review data
1. Read large JSON file in chunks
2. Add sentiment analysis for each record
3. Load into SQL Server

In [8]:
start_time = datetime.now()
file_read_counter = 0

# read the file in chunks, get sentiment analysis and load data into SQL server
for chunk in pd.read_json('yelp_academic_dataset_review.json', lines=True, chunksize=50000):
    file_read_counter += 1
    chunk = chunk[['review_id','business_id','user_id','date','stars','text']]
    chunk['sentiment_analysis'] = chunk['text'].apply(sentiment_analysis)
    chunk.to_sql('yelp_review_data', con=conn, if_exists='append', index=False)

end_time = datetime.now()
elapsed_time = end_time - start_time

print(f'Loop count: {file_read_counter}, Total time taken: {elapsed_time}')

# Process yelp business data
1. Read JSON file in chunks
2. Load into SQL Server

In [13]:
start_time = datetime.now()
file_read_counter = 0

# read the file in chunks, get sentiment analysis and load data into SQL server
for chunk in pd.read_json('yelp_academic_dataset_business.json', lines=True, chunksize=50000):
    file_read_counter += 1
    chunk = chunk[['business_id','name','city','state','review_count','stars','categories']]
    chunk.to_sql('yelp_business_data', con=conn, if_exists='append', index=False)

end_time = datetime.now()
elapsed_time = end_time - start_time

print(f'Loop count: {file_read_counter}, Total time taken: {elapsed_time}')

Loop count: 4, Total time taken: 0:00:25.240662


# Close the connection

In [14]:
conn.close()